# CRAM knowledge playground

Test the repository knowledge system through its real read-only MCP server. Run the **offline inspection** first; it needs no LLM account. The remaining cells support Q&A and code proposals with a provider and model you choose.

Select the **CRAM Knowledge** kernel. Install `repository-knowledge[notebook]` into your configured Python environment if it is not available. See [setup and limits](../docs/notebook.md).

The default `Run All` makes no model request and saves no report. Live requests send your question and model-selected source evidence to your chosen endpoint and can incur charges. Code proposals remain text: this notebook does not edit CRAM files or execute proposed code or tests. Model answers still need review.

Use the ignored local copy under `repository_knowledge/.cache/notebooks/` for interactive sessions. Keep this tracked file as a clean template; the setup guide includes the copy command.

In [ ]:
# %% imports
import os

os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")

from getpass import getpass
from pathlib import Path
from uuid import uuid4

from repository_knowledge.adapters.serialization import RecordCodec
from repository_knowledge.interfaces.notebook import (
    ModelSettings,
    NotebookConnection,
    NotebookSession,
)
from repository_knowledge.model import SourceScope


## Choose the source

The local setup uses an explicitly included working-tree overlay. Its cache must match the selected checkout, HEAD, tracked bytes and untracked inclusion policy. A stale cache is refused; refresh it from the repository root after editing. `CRAM_NOTEBOOK_*` environment overrides also allow isolated fixture runs.

For a committed-only cache, select `SourceScope.COMMITTED`, its database and the matching revision. Working-tree and committed indexes are separate. The notebook's queries do not silently rebuild either one.

In [ ]:
# %% connection
repository_override = os.environ.get("CRAM_NOTEBOOK_REPOSITORY")
repository_root = (
    Path(repository_override).resolve()
    if repository_override
    else NotebookConnection.find_repository(Path.cwd())
)
database_path = Path(
    os.environ.get(
        "CRAM_NOTEBOOK_DATABASE",
        str(repository_root / "repository_knowledge/.cache/agent-working.sqlite3"),
    )
).resolve()
source_scope = SourceScope(
    os.environ.get("CRAM_NOTEBOOK_SCOPE", SourceScope.WORKING_TREE)
)
selected_symbol = os.environ.get(
    "CRAM_NOTEBOOK_SYMBOL", "coraplex.plans.executables.GiskardExecutable"
)
connection = NotebookConnection(
    repository_root=repository_root,
    database_path=database_path,
    source_scope=source_scope,
    revision="HEAD",
    included_untracked_paths=(
        ("repository_knowledge", "test/repository_knowledge_test")
        if source_scope == SourceScope.WORKING_TREE
        else ()
    ),
)
session = NotebookSession(connection)


In [ ]:
# %% offline inspection
inspection = await connection.inspect(selected_symbol)
print("Source commit:", inspection.freshness.indexed_commit)
print("Source scope:", inspection.freshness.source_scope)
print("Snapshot:", inspection.freshness.snapshot_identifier)
print("MCP tools:", inspection.available_tools)
print(RecordCodec().encode(inspection.definition))


## Choose a provider and explicitly enable live calls

Use a [LiteLLM provider identifier](https://docs.litellm.ai/docs/providers) and a model available in your own account, for example a provider identifier such as `openai`, `anthropic` or `ollama`. The adapter constructs `provider/model`; an already prefixed model is accepted. Compatible/local endpoints can use `api_base`.

Set `RUN_LIVE = True`, fill in `MODEL_NAME`, and rerun this cell. Enter the key only in the hidden prompt, never in a saved cell. A local provider may accept an empty key. Credentials stay in kernel memory; restarting the kernel clears them. A provider can also use credentials already configured in your environment. Avoid SDK debug logging and clear outputs before sharing the notebook.

Each live answer has bounded model requests, output and time. Rich `prepare-context` queries currently take about 72–93 seconds on the measured CRAM snapshot, so the repository query deadline is 300 seconds. Start with small symbol queries. `assess-change` requires a server configured with an explicit baseline and is not configured by this notebook.

In [ ]:
# %% provider configuration
RUN_LIVE = False
LLM_PROVIDER = "openai"
MODEL_NAME = ""
API_BASE = None

settings = ModelSettings(
    provider=LLM_PROVIDER,
    model=MODEL_NAME,
    api_base=API_BASE,
    live_enabled=RUN_LIVE,
    maximum_turns=6,
    maximum_output_tokens=2048,
    model_timeout_seconds=60,
    timeout_seconds=300,
)
if settings.live_enabled:
    settings.api_key = (
        getpass("Provider API key (blank for configured/local credentials): ") or None
    )
    print("Selected model:", settings.select_model())
else:
    print("Live model requests are disabled.")


## Ask a source-grounded question

The model chooses bounded repository queries. Source identity is checked before and after the turn, and a changed source refuses the result. The tool descriptions and source receipts distinguish static evidence from runtime certainty. Local wiki prose is not automatically treated as authenticated canonical guidance.

In [ ]:
# %% question and answer
question = (
    f"Explain the responsibilities and direct dependencies of {selected_symbol}. "
    "Start by describing that symbol. Cite the returned file paths and lines, "
    "and identify anything that static evidence does not establish."
)
qa_trial = None
if settings.live_enabled:
    qa_trial = await session.run(question, settings)
    print(qa_trial.answer)
else:
    print("Q&A skipped: enable live requests in the provider cell first.")


## Request a code proposal

Edit this request for a real task. The model can navigate declarations, references, related examples and static pytest evidence. It returns a proposed change plus validation steps. Review the citations, apply any wanted edits in your coding workflow, run the appropriate tests and refresh the overlay before another model turn. No test result is implied by generated test code.

In [ ]:
# %% coding assistance
coding_request = (
    f"For {selected_symbol}, identify one small, useful unit-test opportunity "
    "from its current declaration and nearby test evidence. Inspect existing "
    "examples and applicable instructions. Propose the test code and its file "
    "location, list assumptions and commands for validation, and cite sources. "
    "Do not apply edits or claim that you ran tests."
)
coding_trial = None
if settings.live_enabled:
    coding_trial = await session.run(coding_request, settings)
    print(coding_trial.answer)
else:
    print("Code proposal skipped: live requests are disabled.")


## Inspect usage and query traces

Token totals come from provider/SDK usage when available; `None` means unknown. A positive cost is an SDK estimate, not a billing receipt. Query characters are response sizes, not token counts. Traces omit full source payloads. These observations do not establish correctness, token savings or a successful code change. Compare the same task, model and source against your normal workflow before drawing those conclusions.

In [ ]:
# %% trial receipts
trials = [trial for trial in (qa_trial, coding_trial) if trial is not None]
for trial in trials:
    print("Model:", trial.model, "Stop:", trial.stop_reason)
    print("Source:", trial.source_commit, trial.source_scope, trial.snapshot_identifier)
    print("Seconds:", round(trial.seconds, 3))
    print("Input/output tokens:", trial.input_tokens, trial.output_tokens)
    print("Model calls / tool attempts:", trial.model_calls, trial.tool_calls)
    print("Estimated cost:", trial.estimated_cost)
    for observation in trial.observations:
        print(observation)
if not trials:
    print("No model trials yet; offline MCP inspection is available above.")


## Optional local report

Leave saving disabled unless you want a local experiment receipt. Reports use the typed evidence serializer and exclude provider settings and credentials. They contain model answers and query selections, which can still be sensitive. They are stored under the ignored `.cache/notebook-trials/` directory. The notebook does not save itself or its outputs.

In [ ]:
# %% optional report
SAVE_REPORTS = False
if SAVE_REPORTS and trials:
    report_directory = repository_root / "repository_knowledge/.cache/notebook-trials"
    report_directory.mkdir(parents=True, exist_ok=True)
    for trial in trials:
        report_path = report_directory / f"{uuid4()}.json"
        with report_path.open("x", encoding="utf-8") as report_file:
            report_file.write(RecordCodec().encode(trial))
        print("Saved:", report_path)
else:
    print("Report saving is disabled or no model trials are available.")
